# T34 - 债券新发行

## 第4章：数据处理模块

---

### 本章概述

本章实现债券发行数据的处理模块，包括：n1. 数据清洗和标准化
2. 日期处理
3. 数据验证
4. 数据转换和格式化

### 导入依赖

In [ ]:
# 标准库
import os
import sys
import datetime
from datetime import datetime, timedelta
import re
import warnings

# 数据处理
import pandas as pd
import numpy as np

# 忽略警告
warnings.filterwarnings('ignore')

print("依赖库导入成功")

### 数据清洗函数

In [ ]:
def clean_numeric_value(value):
    """
    清洗数值类型数据
    
    Args:
        value: 原始值
        
    Returns:
        清洗后的数值或None
    """
    if pd.isna(value):
        return None
    if isinstance(value, (int, float)):
        return value if not np.isnan(value) else None
    try:
        # 移除可能的逗号和空格
        cleaned = str(value).replace(',', '').replace(' ', '').strip()
        if cleaned == '' or cleaned == '-':
            return None
        return float(cleaned)
    except (ValueError, TypeError):
        return None


def clean_date_value(value, date_format='%Y-%m-%d'):
    """
    清洗日期类型数据
    
    Args:
        value: 原始值
        date_format: 目标日期格式
        
    Returns:
        清洗后的日期字符串或None
    """
    if pd.isna(value):
        return None
    
    # 如果已经是datetime类型
    if isinstance(value, (datetime, pd.Timestamp)):
        return value.strftime(date_format)
    
    try:
        # 尝试解析日期
        parsed = pd.to_datetime(value, errors='coerce')
        if pd.isna(parsed):
            return None
        return parsed.strftime(date_format)
    except Exception:
        return None


def clean_string_value(value):
    """
    清洗字符串类型数据
    
    Args:
        value: 原始值
        
    Returns:
        清洗后的字符串或None
    """
    if pd.isna(value):
        return None
    cleaned = str(value).strip()
    if cleaned == '' or cleaned == '-' or cleaned == '--':
        return None
    return cleaned


print("数据清洗函数定义完成")

### 数据标准化函数

In [ ]:
def standardize_bond_type(bond_type):
    """
    标准化债券类型名称
    
    Args:
        bond_type: 原始债券类型
        
    Returns:
        标准化后的债券类型
    """
    if pd.isna(bond_type):
        return '其他'
    
    bond_type = str(bond_type).strip()
    
    # 债券类型映射
    type_mapping = {
        '国债': '国债',
        '央行票据': '央行票据',
        '存单': '同业存单',
        '同业存单': '同业存单',
        '普通金融债': '金融债',
        '金融债': '金融债',
        '政策银行债': '政策性金融债',
        '政策性金融债': '政策性金融债',
        '商业银行次级债券': '商业银行次级债',
        '保险债': '保险债',
        '证券公司债': '证券公司债',
        '证券公司短融债': '证券公司短期融资券',
        '其他金融机构债': '其他金融机构债',
        '企业债': '企业债',
        '公司债': '公司债',
        '中期票据': '中期票据',
        '短期融资券': '短期融资券',
        '项目收益票据': '项目收益票据',
        'PPN': '定向债务融资工具',
        'ppn': '定向债务融资工具',
        '定向债务融资工具': '定向债务融资工具',
        '国际机构债': '国际机构债',
        '政府支持机构债': '政府支持机构债',
        '标准化票据': '标准化票据',
        'ABS': 'ABS',
        '可转债': '可转换债券',
        '可转换债券': '可转换债券',
        '可交换债': '可交换债券',
        '可交换债券': '可交换债券',
        '可分离转债': '可分离交易可转债',
    }
    
    return type_mapping.get(bond_type, bond_type)


def standardize_urban_investment_flag(value):
    """
    标准化城投标识
    
    Args:
        value: 原始值
        
    Returns:
        标准化后的值（'是' 或 '否'）
    """
    if pd.isna(value):
        return '否'
    
    value = str(value).strip()
    
    if value in ['是', 'Y', 'y', '1', 'True', 'true', 'YES', 'yes']:
        return '是'
    else:
        return '否'


def standardize_bond_term(term):
    """
    标准化债券期限
    
    Args:
        term: 原始期限值
        
    Returns:
        标准化后的期限（年数）
    """
    if pd.isna(term):
        return None
    
    term_str = str(term).strip()
    
    # 尝试直接转换为数字
    try:
        return float(term_str)
    except ValueError:
        pass
    
    # 解析类似 '3年' '3Y' '3Y6M' 的格式
    years = 0
    
    # 匹配年
    year_match = re.search(r'(\d+)\s*[年Yy]', term_str)
    if year_match:
        years += int(year_match.group(1))
    
    # 匹配月
    month_match = re.search(r'(\d+)\s*[个]?[月Mm]', term_str)
    if month_match:
        years += int(month_match.group(1)) / 12
    
    # 匹配天
    day_match = re.search(r'(\d+)\s*[天Dd]', term_str)
    if day_match:
        years += int(day_match.group(1)) / 365
    
    if years > 0:
        return round(years, 2)
    
    return None


print("数据标准化函数定义完成")

### DataFrame处理函数

In [ ]:
def clean_bond_maturity_data(df):
    """
    清洗债券到期数据
    
    Args:
        df: 原始数据DataFrame
        
    Returns:
        清洗后的DataFrame
    """
    if df is None or df.empty:
        return df
    
    df = df.copy()
    
    # 日期清洗
    if 'dt' in df.columns:
        df['dt'] = df['dt'].apply(clean_date_value)
    
    # 数值清洗
    if 'totalredemption' in df.columns:
        df['totalredemption'] = df['totalredemption'].apply(clean_numeric_value)
    
    # 债券类型标准化
    if 'bondtype' in df.columns:
        df['bondtype'] = df['bondtype'].apply(standardize_bond_type)
    
    # 城投标识标准化
    if 'isurbaninvestmentbonds' in df.columns:
        df['isurbaninvestmentbonds'] = df['isurbaninvestmentbonds'].apply(standardize_urban_investment_flag)
    
    # 删除日期为空的记录
    df = df.dropna(subset=['dt'])
    
    return df


def clean_bond_issue_data(df):
    """
    清洗债券新发行数据
    
    Args:
        df: 原始数据DataFrame
        
    Returns:
        清洗后的DataFrame
    """
    if df is None or df.empty:
        return df
    
    df = df.copy()
    
    # 字符串清洗
    for col in ['trade_code', 'sec_name']:
        if col in df.columns:
            df[col] = df[col].apply(clean_string_value)
    
    # 日期清洗
    if 'dt' in df.columns:
        df['dt'] = df['dt'].apply(clean_date_value)
    
    # 数值清洗
    if 'planissueamount' in df.columns:
        df['planissueamount'] = df['planissueamount'].apply(clean_numeric_value)
    
    # 期限标准化
    if 'bondterm' in df.columns:
        df['bondterm'] = df['bondterm'].apply(standardize_bond_term)
    
    # 债券类型标准化
    if 'bondtype' in df.columns:
        df['bondtype'] = df['bondtype'].apply(standardize_bond_type)
    
    # 城投标识标准化
    if 'isurbaninvestmentbonds' in df.columns:
        df['isurbaninvestmentbonds'] = df['isurbaninvestmentbonds'].apply(standardize_urban_investment_flag)
    
    # 删除关键字段为空的记录
    df = df.dropna(subset=['trade_code', 'sec_name', 'dt'])
    
    return df


print("DataFrame处理函数定义完成")

### 数据验证函数

In [ ]:
def validate_bond_issue_data(df):
    """
    验证债券新发行数据
    
    Args:
        df: 待验证的DataFrame
        
    Returns:
        验证结果字典
    """
    result = {
        'valid': True,
        'total_records': len(df),
        'errors': [],
        'warnings': []
    }
    
    if df is None or df.empty:
        result['valid'] = False
        result['errors'].append('数据为空')
        return result
    
    # 检查必要列
    required_columns = ['trade_code', 'sec_name', 'dt']
    for col in required_columns:
        if col not in df.columns:
            result['errors'].append(f'缺少必要列: {col}')
            result['valid'] = False
    
    # 检查空值
    for col in required_columns:
        if col in df.columns:
            null_count = df[col].isna().sum()
            if null_count > 0:
                result['warnings'].append(f'{col}列有{null_count}个空值')
    
    # 检查日期格式
    if 'dt' in df.columns:
        invalid_dates = df[df['dt'].isna()]
        if len(invalid_dates) > 0:
            result['warnings'].append(f'有{len(invalid_dates)}条记录日期无效')
    
    # 检查发行规模
    if 'planissueamount' in df.columns:
        negative_amounts = df[df['planissueamount'] < 0] if df['planissueamount'].notna().any() else pd.DataFrame()
        if len(negative_amounts) > 0:
            result['warnings'].append(f'有{len(negative_amounts)}条记录发行规模为负数')
    
    # 检查重复
    duplicates = df[df.duplicated(subset=['trade_code'], keep=False)]
    if len(duplicates) > 0:
        result['warnings'].append(f'有{len(duplicates)}条记录债券代码重复')
    
    return result


def validate_bond_maturity_data(df):
    """
    验证债券到期数据
    
    Args:
        df: 待验证的DataFrame
        
    Returns:
        验证结果字典
    """
    result = {
        'valid': True,
        'total_records': len(df),
        'errors': [],
        'warnings': []
    }
    
    if df is None or df.empty:
        result['valid'] = False
        result['errors'].append('数据为空')
        return result
    
    # 检查必要列
    required_columns = ['dt', 'bondtype']
    for col in required_columns:
        if col not in df.columns:
            result['errors'].append(f'缺少必要列: {col}')
            result['valid'] = False
    
    # 检查空值
    if 'dt' in df.columns:
        null_count = df['dt'].isna().sum()
        if null_count > 0:
            result['warnings'].append(f'dt列有{null_count}个空值')
    
    # 检查发行规模
    if 'totalredemption' in df.columns:
        negative_amounts = df[df['totalredemption'] < 0] if df['totalredemption'].notna().any() else pd.DataFrame()
        if len(negative_amounts) > 0:
            result['warnings'].append(f'有{len(negative_amounts)}条记录规模为负数')
    
    return result


print("数据验证函数定义完成")

### 函数测试

In [ ]:
# 测试数值清洗
print("数值清洗测试:")
test_values = ['1,234.56', ' 1000 ', '-', '', None, 123.45, np.nan]
for v in test_values:
    result = clean_numeric_value(v)
    print(f"  '{v}' -> {result}")

In [ ]:
# 测试日期清洗
print("日期清洗测试:")
test_dates = ['2024-01-15', '2024/01/15', '20240115', datetime(2024, 1, 15), None, 'invalid']
for d in test_dates:
    result = clean_date_value(d)
    print(f"  '{d}' -> {result}")

In [ ]:
# 测试期限标准化
print("期限标准化测试:")
test_terms = ['3年', '3Y', '2Y6M', '180天', '5', '3年6个月', None]
for t in test_terms:
    result = standardize_bond_term(t)
    print(f"  '{t}' -> {result}")

In [ ]:
# 测试债券类型标准化
print("债券类型标准化测试:")
test_types = ['国债', '存单', 'PPN', 'ppn', '可转债', '未知类型', None]
for bt in test_types:
    result = standardize_bond_type(bt)
    print(f"  '{bt}' -> {result}")

In [ ]:
# 测试完整的数据清洗流程
test_df = pd.DataFrame({
    'trade_code': ['123456.SH', '  789012.SZ  ', None, '111111.SH'],
    'sec_name': ['测试债券01', '测试债券02', '测试债券03', ''],
    'dt': ['2024-01-15', '2024/01/16', 'invalid', '20240118'],
    'planissueamount': ['1,000', '2,000.5', '-', '500'],
    'bondterm': ['3年', '2Y6M', '180天', None],
    'bondtype': ['企业债', '公司债', 'PPN', 'abs'],
    'isurbaninvestmentbonds': ['是', 'N', 'Y', '否']
})

print("原始数据:")
print(test_df)
print("\n清洗后数据:")
cleaned_df = clean_bond_issue_data(test_df)
print(cleaned_df)

In [ ]:
# 测试数据验证
print("数据验证测试:")
validation_result = validate_bond_issue_data(cleaned_df)
print(f"验证通过: {validation_result['valid']}")
print(f"总记录数: {validation_result['total_records']}")
if validation_result['errors']:
    print(f"错误: {validation_result['errors']}")
if validation_result['warnings']:
    print(f"警告: {validation_result['warnings']}")

---

### 本章小结

**已实现功能**:
- 数值、日期、字符串清洗函数
- 债券类型、城投标识、期限标准化函数
- DataFrame批量处理函数
- 数据验证函数

---

**下一章**: [5_核心逻辑实现](5_核心逻辑实现.ipynb)